# Laboratorio 5: Incisos 1 al 3
*Este notebook contiene la solución a los incisos 1 al 3 del Laboratorio 5*

## Configuración

**Librerías**

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score

**Dependencias**

In [10]:
data = r'C:\Users\ajmac\Documents\Universidad\Noveno semestre\Data Mining\Laboratorio 5\LABORATORIO_5_MD\datos_limpios.csv'
df = pd.read_csv(data)

## Modelo de Regresión con Naive Bayes

In [14]:
# Separar características (X) y objetivo (y)
X = df.drop(columns=['price'])
y = df['price']

# Dividir el conjunto de datos (75% / 25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# Reemplzar valores nan
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)   # fit solo en train
X_test_imp  = imputer.transform(X_test)        # transform en test (sin re-fit)
 
#    Naive Bayes es un clasificador; se mapea
#    cada precio a un intervalo y luego se
#    usa el centro del bin como predicción.
N_BINS = 50
bins = np.linspace(y.min(), y.max() + 1, N_BINS + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
 
y_train_binned = np.clip(np.digitize(y_train, bins) - 1, 0, N_BINS - 1)
 
# Entrenamiento del modelo
model = GaussianNB()
model.fit(X_train_imp, y_train_binned)
 
#    El modelo predice el bin; se convierte
#    al centro del intervalo como valor numérico.
y_pred_bins = model.predict(X_test_imp)
y_pred = bin_centers[y_pred_bins]
 
# Evaluación del modelo
n = len(y_test)
p = X_test_imp.shape[1]
 
mse    = mean_squared_error(y_test, y_pred)
rmse   = np.sqrt(mse)
r2     = r2_score(y_test, y_pred)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
 
print("=" * 40)
print("       MÉTRICAS DE EVALUACIÓN")
print("=" * 40)
print(f"  MSE          : {mse:>15,.4f}")
print(f"  RMSE         : {rmse:>15,.4f}")
print(f"  R²           : {r2:>15,.4f}")
print(f"  R² Ajustado  : {r2_adj:>15,.4f}")
print("=" * 40)
print(f"  Observaciones (test) : {n}")
print(f"  Variables            : {p}")
print("=" * 40)

       MÉTRICAS DE EVALUACIÓN
  MSE          : 33,315,537.1715
  RMSE         :      5,771.9613
  R²           :         -0.8807
  R² Ajustado  :         -0.8875
  Observaciones (test) : 18388
  Variables            : 66


**2. ¿Qué tan bien le fue prediciendo?**

A pesar que el modelo de Naive Bayes no es un modelo de regresión y que dentro de las variables predictoras hay variables con al ta correlación, según los resultados el modelo se desempeñó aceptablemente bien. En base a su RMSE podemos ver que el modelo falla por $5,771.96, que dado el contexto en el que estamos trabajando (precios de propiedades) no parece ser tan grave. Además vale la pena resaltar que la distribución de los precios es bastante dispersa por lo que esto podría afectar al modelo. Esta data no se limpia de 'outliers' porque estos precios son reales y hace sentido porque si hay mansiones en el dataset como casas normales esto contribuye a la alta entropía de los datos. A pesar de su rendimiento regular, se recomienda realizar una validación cruzada para epoder validar la consistencia en los resultados obtenidos en las métricas y consecuentemente en el rendimiento del modelo. 

**3. ¿Qué modelo funcionó mejor?**

El modelo que mejor funcionó fue el Decision Tree con profundidad de raíz de $p$ donde $p$ era la cantidad de variables predictoras. 